In [1]:
import json
from pathlib import Path
import pandas as pd

# Load all cleaned json files from data/interim and merge them
interim_dir = Path("../data/interim")

records = []
for path in sorted(interim_dir.glob("cleaned_*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    #print(f"{path.name}: {len(data)} records")
    records.extend(data)

print(f"\nTotal: {len(records)} episodes")


Total: 14321 episodes


In [2]:
df = pd.DataFrame([
    {
        "title":            r.get("title"),
        "series_name":      r.get("series_name"),
        "episode_number":   r.get("episode_number"),
        "description":      r.get("description"),
        "duration_minutes": r.get("duration_minutes"),
        "release_date":     r.get("release_date"),
        "label":            r.get("label"),
        "cover_url":        r.get("cover_url"),
        "order_number":     r.get("order_number"),
        "source_url":       r.get("source_url"),
        "n_speakers":       len(r.get("speakers") or []),
        "n_genres":         len(r.get("genres") or []),
    }
    for r in records
])

print(f"Shape: {df.shape}")

Shape: (14321, 12)


In [3]:
# Data types and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14321 entries, 0 to 14320
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             14321 non-null  str    
 1   series_name       14321 non-null  str    
 2   episode_number    14321 non-null  int64  
 3   description       14311 non-null  str    
 4   duration_minutes  6224 non-null   float64
 5   release_date      10917 non-null  str    
 6   label             14321 non-null  str    
 7   cover_url         11840 non-null  str    
 8   order_number      7622 non-null   str    
 9   source_url        0 non-null      object 
 10  n_speakers        14321 non-null  int64  
 11  n_genres          14321 non-null  int64  
dtypes: float64(1), int64(3), object(1), str(7)
memory usage: 1.3+ MB


In [4]:
# Missing values — how complete is the data?
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
source_url,14321,100.0
duration_minutes,8097,56.5
order_number,6699,46.8
release_date,3404,23.8
cover_url,2481,17.3
description,10,0.1
title,0,0.0
series_name,0,0.0
episode_number,0,0.0
label,0,0.0


In [5]:
# Episodes without a title — should be 0, otherwise parser issue
no_title = df[df["title"].isnull()]
print(f"Episodes without title: {len(no_title)}")
if len(no_title) > 0:
    print(no_title[["source_url"]].head(10))

Episodes without title: 0


In [6]:
# Series: count and episodes per series
series_counts = df.groupby("series_name").size().sort_values(ascending=False)
print(f"Number of series: {len(series_counts)}\n")
series_counts.head(20)

Number of series: 1279



series_name
(Diverse)                 419
Die Drei ???              337
TKKG                      285
Märchen                   278
John Sinclair             226
Gruselkabinett            211
Benjamin Blümchen         206
Fünf Freunde              188
Bibi Blocksberg           154
Die Drei ??? Kids         126
Karl May                  118
Jan Tenner                117
Midnight Tales            116
Hanni und Nanni           116
Offenbarung 23            111
Teufelskicker             110
Die drei !!!              109
Sherlock Holmes           101
Filme                      99
Europa - Die Originale     97
dtype: int64

In [7]:
# Parse release date and check for problems
df["release_date_parsed"] = pd.to_datetime(
    df["release_date"], format="%d.%m.%Y", errors="coerce"
)

n_missing = df["release_date"].isnull().sum()
n_failed  = df["release_date_parsed"].isnull().sum()
n_errors  = n_failed - n_missing

print(f"No date at all:       {n_missing}")
print(f"Failed to parse:      {n_errors}  ← cleaning candidates")
print(f"Successfully parsed:  {len(df) - n_failed}")

# Which values failed?
df[df["release_date"].notnull() & df["release_date_parsed"].isnull()][
    ["title", "release_date", "source_url"]
].head(20)

No date at all:       3404
Failed to parse:      0  ← cleaning candidates
Successfully parsed:  10917


,title,release_date,source_url


In [8]:
# Releases per year
df["year"] = df["release_date_parsed"].dt.year
df["year"].value_counts().sort_index()

year
1089.0      2
1910.0      1
1912.0      1
1938.0      1
1956.0      1
1960.0      2
1962.0      1
1963.0      1
1969.0      2
1976.0      3
1979.0      9
1980.0     12
1981.0      7
1982.0      2
1983.0      4
1984.0      2
1985.0      2
1986.0      3
1987.0      4
1988.0      8
1989.0      2
1990.0      1
1991.0      6
1992.0     10
1993.0     10
1994.0     13
1995.0     13
1996.0     15
1997.0     27
1998.0     43
1999.0     40
2000.0     71
2001.0    291
2002.0    328
2003.0    321
2004.0    430
2005.0    489
2006.0    683
2007.0    779
2008.0    758
2009.0    692
2010.0    661
2011.0    585
2012.0    408
2013.0    351
2014.0    255
2015.0    299
2016.0    282
2017.0    284
2018.0    263
2019.0    229
2020.0    320
2021.0    443
2022.0    328
2023.0    360
2024.0    318
2025.0    314
2026.0    125
2027.0      1
2028.0      1
Name: count, dtype: int64

In [9]:
# Outliers in duration
print(df["duration_minutes"].describe())

print("\nVery short (< 5 min):")
print(df[df["duration_minutes"] < 5][["title", "series_name", "duration_minutes"]].head(10))

print("\nVery long (> 180 min):")
print(df[df["duration_minutes"] > 180][["title", "series_name", "duration_minutes"]].head(10))

count    6224.000000
mean       58.494762
std        62.338367
min         2.100000
25%        47.082500
50%        58.000000
75%        68.272500
max      4720.000000
Name: duration_minutes, dtype: float64

Very short (< 5 min):
                                    title                     series_name  \
1319                          6. Dezember  Die Drei ??? und der 5. Advent   
1327                         14. Dezember  Die Drei ??? und der 5. Advent   
1329                         16. Dezember  Die Drei ??? und der 5. Advent   
1331                         18. Dezember  Die Drei ??? und der 5. Advent   
1333                         20. Dezember  Die Drei ??? und der 5. Advent   
1334                         21. Dezember  Die Drei ??? und der 5. Advent   
4446               Folge 1 - Die Songs EP                     Eazy Eizbär   
4484    Die gefährliche Spur (Käse-Pizza)              Die Drei ??? Pizza   
4485  Der feuerrote Teufel (Salami-Pizza)              Die Drei ??? Pizza   


In [10]:
# Explore genres and count
genres_series = pd.Series([
    g for r in records for g in (r.get("genres") or [])
])
print(f"Unique genres: {genres_series.nunique()}\n")
genres_series.value_counts().head(20)

Unique genres: 20



Krimi und Detektiv         1241
Grusel und Horror          1108
Abenteuer                  1015
Kinder und Jugend           947
Mystery                     646
Science Fiction             474
Action                      349
Fantasy                     232
Comedy                      173
OTV                         155
Für Mädchen                 124
Wissen und Infotainment     104
Märchen                      92
Liebe und Romantik           91
Western                      80
Thriller                     65
Musik und Lieder             62
Nur ab 18                    42
Pferdegeschichten            40
Für Jungs                    27
Name: count, dtype: int64

In [11]:
# Most active voice actors
all_speakers = pd.Series([
    s["speaker"] for r in records for s in (r.get("speakers") or [])
])
print(f"Total speaker credits: {len(all_speakers)}")
print(f"Unique speakers:       {all_speakers.nunique()}\n")
all_speakers.value_counts().head(20)

Total speaker credits: 99549
Unique speakers:       10527



Rohrbeck, Oliver          602
Mackensy, Lutz            571
Fröhlich, Andreas         482
von der Meden, Andreas    433
Wawrczeck, Jens           419
Draeger, Sascha           390
Paetsch, Hans             362
Schülke, Achim            354
Halver, Konrad            338
Missler, Robert           333
Schneider, Reinhilt       325
Karallus, Thomas          315
Krauss, Helmut            312
Riedel, Lutz              312
Hagen, Till               312
Thormann, Jürgen          310
Lubowski, Manou           306
Schmidt-Foß, Gerrit       305
Bierstedt, Detlef         303
Bach, Patrick             300
Name: count, dtype: int64

In [12]:
# Normalize speaker names for comparison
# Format is "Lastname, Firstname" — let's check for umlaut variants
speakers_df = pd.DataFrame([
    {
        "speaker": s["speaker"],
        "role":    s["role"],
        "title":   r.get("title"),
        "series":  r.get("series_name"),
    }
    for r in records
    for s in (r.get("speakers") or [])
])

print(f"Total speaker credits: {len(speakers_df)}")
print(f"Unique speaker names:  {speakers_df['speaker'].nunique()}")
speakers_df.head(5)

Total speaker credits: 99549
Unique speaker names:  10527


,speaker,role,title,series
0,"Ganser, Marcus",Sprecher des Hörbuchs,No drugs - das ist cool,(Diverse)
1,"Zimmermann, Herbert",Sprecher der Reportage,Weltmeister Deutschland,(Diverse)
2,"Stubel, Wolf-Dieter",Erzähler,Das Geheimnis des Bermuda-Dreiecks,(Diverse)
3,"Genesis, René",Ozeanograph Marlow,Das Geheimnis des Bermuda-Dreiecks,(Diverse)
4,"Schenker, Rolf E.",Captain Kneeshaw,Das Geheimnis des Bermuda-Dreiecks,(Diverse)


In [13]:
# Look for potential umlaut variants of the same person
# Strategy: normalize umlauts and compare against original
umlaut_map = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue",
    "Ä": "Ae", "Ö": "Oe", "Ü": "Ue",
    "ß": "ss",
})

def normalize_name(name: str) -> str:
    return name.translate(umlaut_map).lower().strip()

speakers_df["speaker_normalized"] = speakers_df["speaker"].map(normalize_name)

# Group by normalized name — if a normalized name has more than one
# original spelling, that's a candidate for deduplication
variants = (
    speakers_df.groupby("speaker_normalized")["speaker"]
    .nunique()
    .reset_index()
    .rename(columns={"speaker": "n_variants"})
)

suspicious = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)
print(f"Normalized names with multiple spellings: {len(suspicious)}")
suspicious.head(20)

Normalized names with multiple spellings: 20


,speaker_normalized,n_variants
846,"boehm, franz",2
1757,"doering, alexander",2
9015,"strueven, felix",2
8285,"schroetter, heike",2
8184,"schoenau, marlies",2
7850,"schaelte, max oscar",2
7847,"schaeffler, marco",2
7678,"rueth, michael",2
7547,"roediger, horst",2
6159,"moeseritz, tim",2


In [14]:
# Inspect the actual variants for each suspicious name
for norm_name in suspicious["speaker_normalized"].head(20):
    original_variants = (
        speakers_df[speakers_df["speaker_normalized"] == norm_name]["speaker"]
        .value_counts()
    )
    print(f"\n{norm_name}:")
    print(original_variants.to_string())


boehm, franz:
speaker
Boehm, Franz    3
Böhm, Franz     1

doering, alexander:
speaker
Doering, Alexander    22
Döring, Alexander      8

strueven, felix:
speaker
Strüven, Felix     93
Strueven, Felix     1

schroetter, heike:
speaker
Schrötter, Heike     9
Schroetter, Heike    2

schoenau, marlies:
speaker
Schönau, Marlies     5
Schoenau, Marlies    5

schaelte, max oscar:
speaker
Schälte, Max Oscar     7
Schaelte, Max Oscar    2

schaeffler, marco:
speaker
Schäffler, Marco     4
Schaeffler, Marco    1

rueth, michael:
speaker
Rüth, Michael     9
Rueth, Michael    2

roediger, horst:
speaker
Roediger, Horst    1
Rödiger, Horst     1

moeseritz, tim:
speaker
Moeseritz, Tim    12
Möseritz, Tim      4

maertens, michael:
speaker
Maertens, Michael    7
Märtens, Michael     2

koester, jan:
speaker
Köster, Jan     4
Koester, Jan    3

kaehler, klaus-peter:
speaker
Kaehler, Klaus-Peter    48
Kähler, Klaus-Peter      1

kaehler, klaus-peter:
speaker
Kaehler, Klaus-Peter    48
Kähler, Klaus-

In [15]:
# Check for other common inconsistencies:
# trailing/leading whitespace, double spaces, etc.
has_whitespace_issue = speakers_df["speaker"].str.contains(r"^\s|\s$|\s{2,}", regex=True)
print(f"Names with whitespace issues: {has_whitespace_issue.sum()}")
speakers_df[has_whitespace_issue][["speaker"]].drop_duplicates().head(20)

Names with whitespace issues: 0


,speaker


In [17]:
# Same check for roles
roles_df = speakers_df["role"].value_counts()
print(f"Unique roles: {len(roles_df)}\n")

# Roles that look like they might be the same
# e.g. "Erzähler" vs "Erzaehler" vs "erzähler"
roles_normalized = pd.Series(
    speakers_df["role"].map(normalize_name).value_counts()
)

role_variants = (
    speakers_df.groupby(speakers_df["role"].map(normalize_name))["role"]
    .nunique()
    .reset_index(name="n_variants")  # direkt den neuen Spaltennamen setzen
)

suspicious_roles = role_variants[role_variants["n_variants"] > 1]
print(f"Roles with multiple spellings: {len(suspicious_roles)}")
suspicious_roles.head(20)

Unique roles: 34248

Roles with multiple spellings: 64


,role,n_variants
647,aeltere frau,2
650,aelterer polizist,2
950,aleksa meisner,2
1042,alf,2
1226,alte dame,2
1229,alte frau,2
1254,alter mann,2
2145,ascari da vivo,2
2595,barbara blocksberg,2
4249,caesar,2


In [20]:
# Inspect the actual variants for each suspicious role
for norm_role in suspicious_roles["role"].head(20):
    original_variants = (
        speakers_df[speakers_df["role"].map(normalize_name) == norm_role]["role"]
        .value_counts()
    )
    print(f"\n{norm_role}:")
    print(original_variants.to_string())


aeltere frau:
role
Ältere Frau    2
ältere Frau    1

aelterer polizist:
role
älterer Polizist    1
Älterer Polizist    1

aleksa meisner:
role
Aleksa Meisner    23
aleksa Meisner     1

alf:
role
Alf    16
ALF     5

alte dame:
role
Alte Dame    4
alte Dame    1

alte frau:
role
Alte Frau    23
alte Frau     4

alter mann:
role
Alter Mann    24
alter Mann     1

ascari da vivo:
role
Ascari da Vivo    6
Ascari Da Vivo    1

barbara blocksberg:
role
Barbara Blocksberg    132
Barbara blocksberg      1

caesar:
role
Cäsar     6
Caesar    5

christopher van helsing:
role
Christopher Van Helsing    12
Christopher van Helsing     8

cora:
role
Cora    9
CORA    2

das haessliche junge entlein:
role
Das häßliche junge Entlein    1
das häßliche junge Entlein    1

delroy:
role
Delroy    1
DelRoy    1

der alte d'artagnan:
role
Der alte d'Artagnan    1
der alte d'Artagnan    1

der alte mann:
role
Der alte Mann    2
der alte Mann    1

der grosse wolf:
role
Der große Wolf    2
Der Große Wolf  

In [18]:
# How often does each role appear? Helps spot noise vs real roles
# e.g. one-off typos will have very low counts
role_counts = speakers_df["role"].value_counts()
print("Most common roles:")
print(role_counts.head(20))
print("\nRoles that appear only once (potential noise):")
print(role_counts[role_counts == 1].head(20))

Most common roles:
role
''Unbekannt''            5403
Erzähler                 4379
Polizist                  431
Sprecher des Hörbuchs     401
Erzählerin                347
Bob Andrews               341
Justus Jonas              332
Peter Shaw                331
Credits                   302
Teil der Menge            276
Mutter                    239
George                    231
Mann                      225
Anne                      221
Otto                      215
Kind                      204
Julian                    201
Laura                     199
Dick                      199
Frau                      197
Name: count, dtype: int64

Roles that appear only once (potential noise):
role
Happy Live                                    1
John Morgan                                   1
Horst Weier                                   1
Mark Cooper                                   1
Sabine Bethke                                 1
Götz Klever                                   1
Pit Fröhl

In [19]:
# Summary: what needs fixing before writing to DB?
print("=== Data Quality Summary ===\n")
print(f"Speaker name variants (umlaut issues): {len(suspicious)}")
print(f"Whitespace issues in names:            {has_whitespace_issue.sum()}")
print(f"Role spelling variants:                {len(suspicious_roles)}")
print(f"\nRecords with no speakers at all:       {(df['n_speakers'] == 0).sum()}")
print(f"Records with no genres:                {(df['n_genres'] == 0).sum()}")
print(f"Records with no release date:          {df['release_date'].isnull().sum()}")
print(f"Records with no description:           {df['description'].isnull().sum()}")

=== Data Quality Summary ===

Speaker name variants (umlaut issues): 20
Whitespace issues in names:            0
Role spelling variants:                64

Records with no speakers at all:       6511
Records with no genres:                9121
Records with no release date:          3404
Records with no description:           10
